# 第九章：期权策略组合 — 像搭积木一样拼期权 / Chapter 9: Option Strategies — Building With LEGO

## 你将学到 / What You Will Learn
- 为什么要组合多个期权 / Why you would combine multiple options
- 五种经典策略各适合什么场景 / Five classic strategies and when each one fits
- 怎么读懂策略的收益图 / How to read a strategy's payoff chart

---

## 9.1 为什么不能只买一个？ / 9.1 Why Not Just Buy One?

单独买一个 Call 或 Put，缺点很明显：

A standalone call or put has clear drawbacks:

- 买 Call：如果涨幅不够大，期权费就白花了 / Long call: if the rally is not big enough, the premium is wasted
- 买 Put：如果跌幅不够大，期权费也白花了 / Long put: if the drop is not big enough, the premium is wasted too

怎么办？**把不同的期权组合起来**，可以：

The fix — **combine several options**, which lets you:

- 降低成本 / lower the cost
- 控制风险上限 / cap the risk
- 针对特定的市场判断赚钱 / profit from a specific market view

## 9.2 五种经典策略 / 9.2 Five Classic Strategies

| 策略 / Strategy | 怎么做 / Construction | 适合场景 / Best For | 一句话 / One-liner |
|------|--------|---------|--------|
| **牛市价差 / Bull spread** | 低K买Call + 高K卖Call / Buy low-K call + sell high-K call | 温和看涨 / Mildly bullish | "看涨但不会涨太多" / "Up, but not too far" |
| **熊市价差 / Bear spread** | 高K买Put + 低K卖Put / Buy high-K put + sell low-K put | 温和看跌 / Mildly bearish | "看跌但不会跌太多" / "Down, but not too far" |
| **跨式 / Straddle** | 同K买Call+买Put / Buy both a call and a put at the same K | 大波动但不确定方向 / Big move, direction unknown | "不管涨跌，动就行" / "Any way, as long as it moves" |
| **宽跨式 / Strangle** | 不同K买Call+买Put / Buy call and put at *different* Ks | 大波动(更便宜) / Big move, cheaper version | "跨式的便宜版" / "Straddle on a budget" |
| **蝶式 / Butterfly** | 低K买+中K卖2个+高K买 / Buy low-K, sell 2 middle-K, buy high-K | 窄幅震荡 / Range-bound | "赌价格不怎么动" / "Bet the market stays still" |

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 9.3 互动实验 / 9.3 Interactive Experiment

> 选择一个策略，调整行权价和期权费，观察：
> - **左图**: 组合后的总收益图（绿=赚，红=亏）
> - **右图**: 每个"零件"的收益图（虚线）拼出总收益（实线）
>
> Pick a strategy, tweak the strikes and premiums, and watch:
> - **Left chart**: the combined payoff (green = profit, red = loss)
> - **Right chart**: each leg (dashed) stacking into the combined line (solid)

In [ ]:
def strategy_builder(strategy='Bull Spread (mild bullish)', K1=95, K2=105, prem1=8.0, prem2=3.0):
    plt.close('all')
    S = np.linspace(60, 140, 300)

    builders = {
        'Bull Spread (mild bullish)':
            lambda: (np.maximum(S - K1, 0) - prem1) + (prem2 - np.maximum(S - K2, 0)),
        'Bear Spread (mild bearish)':
            lambda: (np.maximum(K2 - S, 0) - prem2) + (prem1 - np.maximum(K1 - S, 0)),
        'Straddle (big move)':
            lambda: (np.maximum(S - K1, 0) - prem1) + (np.maximum(K1 - S, 0) - prem2),
        'Strangle (cheaper big move)':
            lambda: (np.maximum(S - K2, 0) - prem2) + (np.maximum(K1 - S, 0) - prem1),
        'Butterfly (sideways)':
            lambda: (np.maximum(S - K1, 0) - prem1)
                  - 2 * (np.maximum(S - 100, 0) - (prem1 + prem2) / 2)
                  + (np.maximum(S - K2, 0) - prem2),
    }
    pnl = builders[strategy]()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # Left: total payoff
    ax = axes[0]
    ax.fill_between(S, pnl, 0, where=(pnl >= 0), color=C_GREEN, alpha=0.3, label='Profit')
    ax.fill_between(S, pnl, 0, where=(pnl < 0), color=C_RED, alpha=0.3, label='Loss')
    ax.plot(S, pnl, 'k-', lw=3)
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    ax.axvline(x=K1, color=C_BLUE, ls='--', alpha=0.4, label=f'K1={K1}')
    ax.axvline(x=K2, color=C_PURPLE, ls='--', alpha=0.4, label=f'K2={K2}')
    ax.annotate(f'Max profit: ${pnl.max():.1f}', xy=(S[np.argmax(pnl)], pnl.max()),
                xytext=(S[np.argmax(pnl)] + 5, pnl.max() + 2),
                fontsize=10, fontweight='bold', color=C_GREEN,
                arrowprops=dict(arrowstyle='->', color=C_GREEN, lw=1.5))
    ax.annotate(f'Max loss: ${pnl.min():.1f}', xy=(S[np.argmin(pnl)], pnl.min()),
                xytext=(S[np.argmin(pnl)] + 5, pnl.min() - 3),
                fontsize=10, fontweight='bold', color=C_RED,
                arrowprops=dict(arrowstyle='->', color=C_RED, lw=1.5))
    ax.set_xlabel('Spot at expiry', fontsize=11)
    ax.set_ylabel('P&L', fontsize=11)
    ax.set_title(strategy, fontsize=14, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Right: legs decomposition
    ax = axes[1]
    if 'Bull' in strategy:
        leg1 = np.maximum(S - K1, 0) - prem1
        leg2 = prem2 - np.maximum(S - K2, 0)
        ax.plot(S, leg1, C_GREEN, ls='--', lw=1.5, label=f'Long Call K={K1} (pay ${prem1})')
        ax.plot(S, leg2, C_RED, ls='--', lw=1.5, label=f'Short Call K={K2} (recv ${prem2})')
    elif 'Bear' in strategy:
        leg1 = np.maximum(K2 - S, 0) - prem2
        leg2 = prem1 - np.maximum(K1 - S, 0)
        ax.plot(S, leg1, C_RED, ls='--', lw=1.5, label=f'Long Put K={K2} (pay ${prem2})')
        ax.plot(S, leg2, C_GREEN, ls='--', lw=1.5, label=f'Short Put K={K1} (recv ${prem1})')
    elif 'Straddle' in strategy:
        leg1 = np.maximum(S - K1, 0) - prem1
        leg2 = np.maximum(K1 - S, 0) - prem2
        ax.plot(S, leg1, C_GREEN, ls='--', lw=1.5, label=f'Long Call K={K1}')
        ax.plot(S, leg2, C_RED, ls='--', lw=1.5, label=f'Long Put K={K1}')
    elif 'Strangle' in strategy:
        leg1 = np.maximum(S - K2, 0) - prem2
        leg2 = np.maximum(K1 - S, 0) - prem1
        ax.plot(S, leg1, C_GREEN, ls='--', lw=1.5, label=f'Long Call K={K2}')
        ax.plot(S, leg2, C_RED, ls='--', lw=1.5, label=f'Long Put K={K1}')
    else:
        leg1 = np.maximum(S - K1, 0) - prem1
        leg2 = np.maximum(S - K2, 0) - prem2
        ax.plot(S, leg1, C_GREEN, ls='--', lw=1.5, label='Wing 1')
        ax.plot(S, leg2, C_RED, ls='--', lw=1.5, label='Wing 2')

    ax.plot(S, pnl, 'k-', lw=3, label='Combined')
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    ax.set_xlabel('Spot at expiry', fontsize=11)
    ax.set_ylabel('P&L', fontsize=11)
    ax.set_title('Legs -> Combined', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    net_cost = prem1 - prem2 if ('Bull' in strategy or 'Bear' in strategy) else prem1 + prem2
    print(f'  Net cost: ${net_cost:.1f}')
    print(f'  Max profit: ${pnl.max():.1f} | Max loss: ${pnl.min():.1f}')

interact(strategy_builder,
         strategy=Dropdown(options=['Bull Spread (mild bullish)',
                                    'Bear Spread (mild bearish)',
                                    'Straddle (big move)',
                                    'Strangle (cheaper big move)',
                                    'Butterfly (sideways)'],
                           description='Strategy:'),
         K1=FloatSlider(value=95, min=70, max=110, step=1, description='K1(low):'),
         K2=FloatSlider(value=105, min=90, max=140, step=1, description='K2(high):'),
         prem1=FloatSlider(value=8.0, min=1, max=20, step=0.5, description='Prem 1($):'),
         prem2=FloatSlider(value=3.0, min=0.5, max=15, step=0.5, description='Prem 2($):'));

## 9.4 小结 / 9.4 Chapter Recap

> 核心思路：**通过组合不同期权，定制你想要的风险-收益形状**
>
> Core idea: **combine options to sculpt the exact risk / reward shape you want.**
>
> - 温和看涨 → 牛市价差（省钱，但盈利有上限）/ Mildly bullish → bull spread (cheaper, but capped upside)
> - 温和看跌 → 熊市价差 / Mildly bearish → bear spread
> - "不知道涨跌，但觉得会大动" → 跨式/宽跨式 / "No view on direction, but big move coming" → straddle or strangle
> - "觉得市场会很平静" → 蝶式 / "Market will stay calm" → butterfly

> **下一章：利率互换** —— 两家公司怎么交换各自的利率风险？
>
> **Next chapter: interest-rate swaps** — how two companies swap their rate exposures.